# Basic WSSS pipeline

Just dino + decoder

In [ ]:
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from dataset import (
    make_voc_datasets, 
    VOC_CLASSES, 
    colorize, 
    denorm_to_uint8,
    get_wsss_dataloaders,
    cache_dino_tokens
)

ModuleNotFoundError: No module named 'pydensecrf'

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading DINOv3 (ViT-Small)...")
dino_repo_dir = "./dinov3"
dino_model = torch.hub.load(
    dino_repo_dir,
    "dinov3_vits16",
    source="local",
    weights="weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth",
).to(device)
dino_model.eval()

train_ds, val_ds = make_voc_datasets(root="./data", resize_size=IMG_SIZE)

train_loader, train_cache_loader, val_loader = get_wsss_dataloaders(
    train_ds=train_ds, 
    val_ds=val_ds, 
    batch_size=BATCH_SIZE, 
    num_workers=NUM_WORKERS
)

## Classifier head

Frozen DINOv3 -> GAP over patch tokens

In [ ]:
NUM_CLASSES = 20
EMBED_DIM = dino_model.embed_dim
PATCH_SIZE = dino_model.patch_size
GRID = IMG_SIZE // PATCH_SIZE
NUM_PATCHES = GRID * GRID


class CAMClassifier(nn.Module):
    """
    A single linear classifier
    """

    def __init__(self, embed_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(embed_dim, num_classes, bias=True)

    def forward(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        pooled = patch_tokens.mean(dim=1)
        return self.fc(pooled)

    @torch.no_grad()
    def cam(self, patch_tokens: torch.Tensor, grid: int) -> torch.Tensor:
        """
        Raw CAM on the patch grid: [B, C, grid, grid] (no normalization).
        Similar to Zhou et al., CVPR 2016
        """
        B, P, D = patch_tokens.shape
        feats = patch_tokens.transpose(1, 2).reshape(B, D, grid, grid)
        W = self.fc.weight
        cams = F.conv2d(feats, W.unsqueeze(-1).unsqueeze(-1))
        return cams
        
    @torch.no_grad()
    def predict(self, patch_tokens: torch.Tensor, grid: int, target_size: tuple[int, int], image_labels: torch.Tensor, bg_threshold: float = 0.25):
        """
        Produce a discrete mask via ReLU, min-max normalization, gating by label, and argmaxing against background.
        Returns:
            cams: [B, C, H, W] normalized CAM mask
            preds: [B, H, W] integer predictions (0..20)
        """
        cams = self.cam(patch_tokens.float(), grid)
        cams = F.relu(cams)
        cams = F.interpolate(cams, size=target_size, mode="bilinear", align_corners=False)

        B, C, H, W = cams.shape
        flat = cams.view(B, C, -1)
        cmin = flat.min(dim=2, keepdim=True).values
        cmax = flat.max(dim=2, keepdim=True).values
        flat = (flat - cmin) / (cmax - cmin).clamp(min=1e-6)
        cams = flat.view_as(cams)

        mask = image_labels.view(B, C, 1, 1)
        cams = cams * mask

        bg = torch.full((B, 1, H, W), bg_threshold, device=cams.device, dtype=cams.dtype)
        full = torch.cat([bg, cams], dim=1)
        preds = full.argmax(dim=1)
        
        return cams, preds

classifier = CAMClassifier(EMBED_DIM, NUM_CLASSES).to(device)
print(classifier)

## Cache DINO patch tokens

DINO is frozen, so we extract patch tokens once and keep them on device

In [ ]:
train_tokens, train_labels = cache_dino_tokens(
    dino_model, train_ds, train_cache_loader, device=device
)
val_tokens, val_labels = cache_dino_tokens(
    dino_model, val_ds, val_loader, device=device
)

## Train the classifier

NN things

In [ ]:
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = AdamW(classifier.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
bce = nn.BCEWithLogitsLoss()


def cached_batches(tokens, labels, batch_size, shuffle=True):
    n = tokens.shape[0]
    order = torch.randperm(n, device=tokens.device) if shuffle else torch.arange(n, device=tokens.device)
    for i in range(0, n, batch_size):
        idx = order[i : i + batch_size]
        yield tokens[idx].float(), labels[idx]


@torch.no_grad()
def eval_classifier(tokens, labels, batch_size=256):
    classifier.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in cached_batches(tokens, labels, batch_size, shuffle=False):
        logits = classifier(x)
        loss_sum += bce(logits, y).item() * x.size(0)
        pred = (logits.sigmoid() > 0.5).float()
        correct += ((pred == y).float().mean(dim=1) == 1).sum().item()
        total += x.size(0)
    return loss_sum / total, correct / total


history = {"train_loss": [], "val_loss": [], "val_exact": []}
for epoch in range(1, EPOCHS + 1):
    classifier.train()
    epoch_loss, n_seen = 0.0, 0
    for x, y in cached_batches(train_tokens, train_labels, BATCH_SIZE, shuffle=True):
        optimizer.zero_grad()
        loss = bce(classifier(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
        n_seen += x.size(0)
    scheduler.step()

    train_loss = epoch_loss / n_seen
    val_loss, val_exact = eval_classifier(val_tokens, val_labels)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_exact"].append(val_exact)
    print(f"epoch {epoch:02d} | train BCE {train_loss:.4f} "
          f"| val BCE {val_loss:.4f} | val exact-match {val_exact:.3f}")

## Visualize CAMs

In [ ]:
def visualize(num_images: int = 3, seed: int = 0):
    rng = random.Random(seed)
    indices = rng.sample(range(len(val_ds)), num_images)

    fig, axes = plt.subplots(num_images, 3, figsize=(12, 4 * num_images))
    if num_images == 1:
        axes = [axes]

    for row, idx in zip(axes, indices):
        _, img_t, label_vec, _ = val_ds[idx]
        img_batch = img_t.unsqueeze(0).to(device)
        label_batch = label_vec.unsqueeze(0).to(device)
        tokens = val_tokens[idx].unsqueeze(0).float()

        cams, raw_labels = classifier.predict(tokens, GRID, target_size=(IMG_SIZE, IMG_SIZE), image_labels=label_batch)
        raw_labels = raw_labels[0].cpu().numpy()

        present = [VOC_CLASSES[c] for c in torch.where(label_vec > 0)[0].tolist()]
        cam_vis = cams[0].max(dim=0).values.cpu().numpy()

        row[0].imshow(denorm_to_uint8(img_t)); row[0].set_title(",".join(present) or "(none)")
        row[1].imshow(cam_vis, cmap="jet", vmin=0, vmax=1); row[1].set_title("max CAM")
        row[2].imshow(colorize(raw_labels)); row[2].set_title("predicted mask")
        for ax in row:
            ax.axis("off")
    plt.tight_layout(); plt.show()


visualize(num_images=3)

## Validation mIoU

Score per-class IoU against ground-truth VOC masks for the raw CAM-argmax masks.

In [ ]:
from dataset import evaluate_masks

def evaluate_miou(max_batches: int | None = None):
    n_cls = NUM_CLASSES + 1
    inter = np.zeros(n_cls, dtype=np.int64)
    union = np.zeros(n_cls, dtype=np.int64)

    loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=wsss_collate_fn,
    )

    for b_idx, batch in enumerate(tqdm(loader, desc="mIoU")):
        if max_batches is not None and b_idx >= max_batches:
            break
            
        indices, images, labels, masks = batch
        
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        tokens = val_tokens[indices].float()

        _, preds = classifier.predict(tokens, GRID, target_size=(IMG_SIZE, IMG_SIZE), image_labels=labels)

        for b, _ in enumerate(indices.tolist()):
            mask_t = masks[b].to(device)
            H, W = mask_t.shape
            
            # Predict method gives [B, IMG_SIZE, IMG_SIZE], but the actual GT masks might be arbitrary shapes. 
            # We resize the prediction to the original GT mask shape.
            pred_b = preds[b:b+1].float().unsqueeze(0)
            p = F.interpolate(pred_b, size=(H, W), mode="nearest-exact").squeeze(0).squeeze(0)
            
            # Use standard evaluation from dataset.py
            b_int, b_uni, _ = evaluate_masks(p, mask_t, num_classes=n_cls, ignore_index=255)
            inter += b_int.cpu().numpy().astype(np.int64)
            union += b_uni.cpu().numpy().astype(np.int64)

    iou = inter / np.maximum(union, 1)
    present = union > 0
    miou = iou[present].mean()

    print(f"\nRaw CAM mIoU = {miou:.4f}")
    for c, name in enumerate(["background"] + VOC_CLASSES):
        if present[c]:
            print(f"  {name:<14s} {iou[c]:.4f}")
    return miou

print("\nEvaluating Raw CAMs:")
evaluate_miou()